# Demo — Personal Enrollment + Patch Generation

Enroll yourself as a new identity, run CRISE to find your high-saliency regions,
then generate a printable occlusion patch targeting those regions.

**Steps:**
1. Place 5–10 photos of your face in `data/demo_identity/{your_name}/`
2. Run this notebook — it enrolls you, runs CRISE, generates a printable patch
3. Validate digitally: apply the patch to your probe images and confirm rank-1 drops
4. Print the patch, wear it, demo live against `demo_live.py`

**The patch is personalized** — derived from which regions *this specific model*
relies on for *your specific face*. Not a generic occlusion.

In [ ]:
# ---------------------------------------------------------------------------
# Config — SET THESE BEFORE RUNNING
# ---------------------------------------------------------------------------

YOUR_NAME     = "your_name_here"   # ← change this (no spaces, use underscore)
PHOTO_DIR     = f"data/demo_identity/{YOUR_NAME}"   # put your photos here

GALLERY_EMB   = "cache/G.npy"
GALLERY_IDS   = "cache/gallery_ids.npy"

CRISE_OUT_DIR = "results/crise_maps"
PATCH_DIR     = f"results/demo_patch/{YOUR_NAME}"

# CRISE hyperparams
N             = 1000
S             = 8
P1            = 0.5
TAU           = 0.1
RUN_SEED      = 9999

# Patch config
PATCH_BUDGET  = 0.15    # mask top 15% of pixels
PATCH_COLOUR  = (0, 0, 0)   # black occlusion (BGR) — change to (255,255,255) for white

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from insightface.app import FaceAnalysis

from rise import build_aligned_chip_112, get_embedding_from_chip
from crise import run_crise

os.makedirs(PATCH_DIR, exist_ok=True)
os.makedirs(CRISE_OUT_DIR, exist_ok=True)

assert os.path.isdir(PHOTO_DIR), (
    f"Photo directory not found: {PHOTO_DIR}\n"
    f"Create it and add 5-10 photos of your face."
)

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("InsightFace ready")

In [ ]:
# ---------------------------------------------------------------------------
# Load all photos, pick gallery image (first), rest are probes
# ---------------------------------------------------------------------------

photo_paths = sorted([
    os.path.join(PHOTO_DIR, f)
    for f in os.listdir(PHOTO_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

assert len(photo_paths) >= 2, "Need at least 2 photos (1 gallery + 1 probe)"

gallery_path = photo_paths[0]
probe_paths  = photo_paths[1:]

print(f"Gallery image : {os.path.basename(gallery_path)}")
print(f"Probe images  : {len(probe_paths)}")

In [ ]:
# ---------------------------------------------------------------------------
# Compute gallery embedding for YOUR_NAME
# Augment the existing gallery with your embedding (in memory only)
# ---------------------------------------------------------------------------

G_base      = np.load(GALLERY_EMB).astype(np.float32)          # (1680, 512)
gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()

gallery_img = cv2.imread(gallery_path)
gallery_chip = build_aligned_chip_112(gallery_img, app)
your_emb    = get_embedding_from_chip(gallery_chip, rec).reshape(1, -1)

G_demo      = np.vstack([G_base, your_emb]).astype(np.float32)
ids_demo    = gallery_ids + [YOUR_NAME]
id_to_index = {gid: i for i, gid in enumerate(ids_demo)}

print(f"Gallery size: {G_base.shape[0]} → {G_demo.shape[0]} (added {YOUR_NAME})")
print(f"Your embedding norm: {np.linalg.norm(your_emb):.4f}")

# Verify: your gallery image should rank-1 match itself
sims     = G_demo @ your_emb.ravel()
rank1_id = ids_demo[int(np.argmax(sims))]
print(f"Gallery self-match: {rank1_id} (should be {YOUR_NAME})")

In [ ]:
# ---------------------------------------------------------------------------
# Verify probe images rank-1 match YOU before patching
# ---------------------------------------------------------------------------

probe_results = []
for path in probe_paths:
    img = cv2.imread(path)
    if img is None:
        print(f"[skip] could not read {path}")
        continue
    try:
        chip = build_aligned_chip_112(img, app)
        emb  = get_embedding_from_chip(chip, rec)
        sims = G_demo @ emb
        rank1 = ids_demo[int(np.argmax(sims))]
        sim   = float(sims[id_to_index[YOUR_NAME]])
        probe_results.append({"path": path, "rank1": rank1, "sim": sim,
                               "correct": rank1 == YOUR_NAME, "chip": chip})
        print(f"  {os.path.basename(path):30s}  rank1={rank1:25s}  sim={sim:.3f}  {'✓' if rank1==YOUR_NAME else '✗'}")
    except Exception as e:
        print(f"[fail] {path}: {e}")

n_correct = sum(r["correct"] for r in probe_results)
print(f"\nBaseline rank-1: {n_correct}/{len(probe_results)} probes correct")

In [ ]:
# ---------------------------------------------------------------------------
# Run CRISE on your probe images
# ---------------------------------------------------------------------------

crise_results = []
for k, path in enumerate(tqdm(probe_paths, desc="Running CRISE")):
    img = cv2.imread(path)
    if img is None:
        continue
    try:
        result = run_crise(
            true_id     = YOUR_NAME,
            probe_path  = path,
            G           = G_demo,
            id_to_index = id_to_index,
            rec         = rec,
            app         = app,
            run_seed    = RUN_SEED + k,
            out_dir     = CRISE_OUT_DIR,
            tau         = TAU,
            N           = N,
            s           = S,
            p1          = P1,
        )
        crise_results.append(result)
        print(f"  {os.path.basename(path)} — w_clean={result['w_clean']:.3f}")
    except Exception as e:
        print(f"[fail] {path}: {e}")

print(f"\nCRISE complete: {len(crise_results)} maps")

In [ ]:
# ---------------------------------------------------------------------------
# Build mean saliency map across all your probes
# ---------------------------------------------------------------------------

sal_maps = [r["sal_norm"] for r in crise_results if r["sal_norm"] is not None]
assert len(sal_maps) > 0, "No saliency maps computed"

mean_sal = np.mean(sal_maps, axis=0).astype(np.float32)

# Save mean map
mean_sal_path = os.path.join(PATCH_DIR, f"{YOUR_NAME}_mean_saliency.npy")
np.save(mean_sal_path, mean_sal)

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(cv2.cvtColor(crise_results[0]["chip_bgr"], cv2.COLOR_BGR2RGB))
axes[0].set_title("Your aligned chip")
axes[1].imshow(mean_sal, cmap="hot", vmin=0, vmax=1)
axes[1].set_title(f"Mean CRISE saliency ({len(sal_maps)} probes)")
axes[2].imshow(cv2.cvtColor(crise_results[0]["chip_bgr"], cv2.COLOR_BGR2RGB))
axes[2].imshow(mean_sal, alpha=0.55, cmap="hot", vmin=0, vmax=1)
axes[2].set_title("Overlay")
for ax in axes: ax.axis("off")
plt.suptitle(f"CRISE saliency — {YOUR_NAME}", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(PATCH_DIR, "saliency_overview.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Generate printable patch
# Top PATCH_BUDGET% pixels highlighted as occlusion regions on a white background
# ---------------------------------------------------------------------------

k_pixels   = int(112 * 112 * PATCH_BUDGET)
flat_sal   = mean_sal.ravel()
top_k_idx  = np.argsort(flat_sal)[-k_pixels:]

# Binary mask on 112×112
patch_mask = np.zeros((112, 112), dtype=np.uint8)
ys, xs     = np.unravel_index(top_k_idx, (112, 112))
patch_mask[ys, xs] = 255

# Upscale to printable size (×8 → 896×896 px ≈ A5 at 150dpi)
SCALE         = 8
patch_large   = cv2.resize(patch_mask, (112 * SCALE, 112 * SCALE),
                           interpolation=cv2.INTER_NEAREST)

# Colour: black occlusion on white background
patch_colour  = np.ones((112 * SCALE, 112 * SCALE, 3), dtype=np.uint8) * 255
patch_colour[patch_large == 255] = list(PATCH_COLOUR)

patch_path = os.path.join(PATCH_DIR, f"{YOUR_NAME}_patch_{int(PATCH_BUDGET*100)}pct.png")
cv2.imwrite(patch_path, patch_colour)
print(f"Printable patch saved: {patch_path}")
print(f"  Size: {patch_colour.shape[1]}×{patch_colour.shape[0]} px")
print(f"  Pixels masked: {k_pixels} / {112*112} ({PATCH_BUDGET*100:.0f}%)")

# Preview
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cv2.cvtColor(patch_colour, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Printable patch ({PATCH_BUDGET*100:.0f}% coverage)")
axes[1].imshow(patch_mask, cmap="gray")
axes[1].set_title("Patch mask (112×112)")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(PATCH_DIR, "patch_preview.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Digital validation: apply patch to probe chips, measure rank-1 drop
# ---------------------------------------------------------------------------

BUDGETS_VAL = [0.05, 0.10, 0.15, 0.20, 0.30]

val_results = []
for pr in probe_results:
    chip = pr["chip"]
    row  = {"probe": os.path.basename(pr["path"]), "clean": pr["correct"]}

    for b in BUDGETS_VAL:
        k  = int(112 * 112 * b)
        tk = np.argsort(mean_sal.ravel())[-k:]
        masked = chip.copy().astype(np.float32)
        mean_c = masked.mean(axis=(0, 1))
        y_, x_ = np.unravel_index(tk, (112, 112))
        masked[y_, x_] = mean_c
        masked = masked.astype(np.uint8)
        try:
            emb   = get_embedding_from_chip(masked, rec)
            sims  = G_demo @ emb
            row[f"b{int(b*100)}"] = (ids_demo[int(np.argmax(sims))] == YOUR_NAME)
        except Exception:
            row[f"b{int(b*100)}"] = None

    val_results.append(row)

val_df = pd.DataFrame(val_results)
print(val_df.to_string(index=False))

# Rank-1 accuracy per budget
print("\nRank-1 accuracy after patch:")
for b in BUDGETS_VAL:
    col = f"b{int(b*100)}"
    if col in val_df.columns:
        acc = val_df[col].mean()
        print(f"  {int(b*100):3d}%: {acc:.3f}")

In [ ]:
# ---------------------------------------------------------------------------
# Visualise patched chips
# ---------------------------------------------------------------------------

chip0 = probe_results[0]["chip"]

fig, axes = plt.subplots(1, len(BUDGETS_VAL) + 1, figsize=(3 * (len(BUDGETS_VAL) + 1), 3))
axes[0].imshow(cv2.cvtColor(chip0, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")

for i, b in enumerate(BUDGETS_VAL):
    k  = int(112 * 112 * b)
    tk = np.argsort(mean_sal.ravel())[-k:]
    masked = chip0.copy().astype(np.float32)
    mean_c = masked.mean(axis=(0, 1))
    y_, x_ = np.unravel_index(tk, (112, 112))
    masked[y_, x_] = mean_c
    axes[i + 1].imshow(cv2.cvtColor(masked.astype(np.uint8), cv2.COLOR_BGR2RGB))
    acc = val_df[f"b{int(b*100)}"].mean() if f"b{int(b*100)}" in val_df.columns else float("nan")
    axes[i + 1].set_title(f"{int(b*100)}% masked\nacc={acc:.2f}")

for ax in axes: ax.axis("off")
plt.suptitle(f"Digital patch validation — {YOUR_NAME}", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PATCH_DIR, "digital_validation.png"), dpi=150, bbox_inches="tight")
plt.show()